[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](http://colab.research.google.com/github/masanorihirano/pams/blob/main/examples/dark_pool.ipynb)

In [ ]:
# Please remove comment-out if necessary
#! pip install pams matplotlib

# Dark pool

This example is a port of the DarkPool sample of plhamJ, which is based on Mizuta et al. (2015).
FCN agents trade in a lit market and in a dark pool beside it.
Each agent decides its order to the lit market as `FCNAgent` does, and sends the order to the dark pool instead, as a market order, with the probability `darkPoolChance`.
The dark pool does not show its order book and executes its orders in time priority at the mid price of the lit market.

The same classes are in `samples/dark_pool` of the PAMS repository, and `samples/dark_pool/main.py` prints the same columns as plhamJ.

Reference:
- Mizuta, T., Kosugi, S., Kusumoto, T., Matsumoto, W., & Izumi, K. (2015). Effects of dark pools on financial markets' efficiency and price discovery function: an investigation by multi-agent simulations. Evolutionary and Institutional Economics Review, 12(2), 375–394. https://doi.org/10.1007/s40844-015-0020-3

In [ ]:
import copy
import random
from typing import Any
from typing import Dict
from typing import List
from typing import Optional
from typing import Union
from typing import cast

import matplotlib.pyplot as plt
import numpy as np

from pams.agents import FCNAgent
from pams.logs import ExecutionLog
from pams.logs import MarketStepEndLog
from pams.logs import OrderLog
from pams.logs.market_step_loggers import MarketStepSaver
from pams.market import Market
from pams.order import MARKET_ORDER
from pams.order import Cancel
from pams.order import Order
from pams.runners import SequentialRunner
from pams.utils import JsonRandom

## Dark pool market

`DarkPoolMarket` accepts only market orders.
Its lit market is set by the parameter `markets`.
The best buy and sell orders are matched in time priority at the current mid price of the lit market, or at the market price of the lit market if one side of the order book of the lit market is empty.
The price is rounded to the tick size in favor of the earlier order.
The dark pool cancels the orders placed while it is not running, that is, in the session without order execution.

In [ ]:
class DarkPoolMarket(Market):
    lit_market: Market

    def setup(self, settings: Dict[str, Any], *args: Any, **kwargs: Any) -> None:
        super().setup(settings, *args, **kwargs)
        if "markets" not in settings:
            raise ValueError("markets is required for DarkPoolMarket as the lit market")
        market_names: Any = settings["markets"]
        if not isinstance(market_names, list) or len(market_names) != 1:
            raise ValueError("markets of DarkPoolMarket must have exactly one market")
        if market_names[0] not in self.simulator.name2market:
            raise ValueError(f"lit market {market_names[0]} does not exist")
        lit_market: Market = self.simulator.name2market[market_names[0]]
        if isinstance(lit_market, DarkPoolMarket):
            raise ValueError("the lit market of DarkPoolMarket must not be a dark pool")
        self.lit_market = lit_market

    def get_lit_mid_price(self) -> float:
        best_buy_price: Optional[float] = self.lit_market.get_best_buy_price()
        best_sell_price: Optional[float] = self.lit_market.get_best_sell_price()
        if best_buy_price is None or best_sell_price is None:
            return self.lit_market.get_market_price()
        return (best_buy_price + best_sell_price) / 2.0

    def _add_order(self, order: Order) -> OrderLog:
        if order.kind != MARKET_ORDER:
            raise ValueError("DarkPoolMarket accepts only market orders")
        log: OrderLog = super()._add_order(order=order)
        if not self.is_running:
            self._cancel_order(cancel=Cancel(order=order))
        return log

    def remain_executable_orders(self) -> bool:
        return len(self.buy_order_book) > 0 and len(self.sell_order_book) > 0

    def _execution(self) -> List[ExecutionLog]:
        logs: List[ExecutionLog] = []
        lit_mid_price: float = self.get_lit_mid_price()
        while self.remain_executable_orders():
            buy_order: Order = cast(Order, self.buy_order_book.get_best_order())
            sell_order: Order = cast(Order, self.sell_order_book.get_best_order())
            is_buy_later: bool = cast(int, buy_order.order_id) > cast(int, sell_order.order_id)
            # rounding as a sell price (up) favors the earlier sell order and rounding
            # as a buy price (down) favors the earlier buy order.
            price: float = self.convert_to_price(
                tick_level=self.convert_to_tick_level(price=lit_mid_price, is_buy=not is_buy_later)
            )
            volume: int = min(buy_order.volume, sell_order.volume)
            logs.append(
                self._execute_orders(price=price, volume=volume, buy_order=buy_order, sell_order=sell_order)
            )
        return logs

## Dark pool FCN agent

`DarkPoolFCNAgent` orders only when it is asked for the orders to a dark pool.
It then decides the orders to the lit market of the dark pool as `FCNAgent` does, and sends each of them to the dark pool instead with the probability `darkPoolChance`.

In [ ]:
class DarkPoolFCNAgent(FCNAgent):
    dark_pool_chance: float

    def setup(
        self, settings: Dict[str, Any], accessible_markets_ids: List[int], *args: Any, **kwargs: Any
    ) -> None:
        super().setup(settings=settings, accessible_markets_ids=accessible_markets_ids)
        if "darkPoolChance" not in settings:
            raise ValueError("darkPoolChance is required for DarkPoolFCNAgent")
        self.dark_pool_chance = JsonRandom(prng=self.prng).random(json_value=settings["darkPoolChance"])
        if not 0.0 <= self.dark_pool_chance <= 1.0:
            raise ValueError("darkPoolChance must be between 0.0 and 1.0")

    def submit_orders_by_market(self, market: Market) -> List[Union[Order, Cancel]]:
        if not isinstance(market, DarkPoolMarket):
            return []
        if not self.is_market_accessible(market_id=market.market_id):
            return []
        orders: List[Union[Order, Cancel]] = []
        for lit_order in super().submit_orders_by_market(market=market.lit_market):
            if isinstance(lit_order, Order) and self.prng.random() < self.dark_pool_chance:
                orders.append(
                    Order(
                        agent_id=self.agent_id,
                        market_id=market.market_id,
                        is_buy=lit_order.is_buy,
                        kind=MARKET_ORDER,
                        volume=lit_order.volume,
                        ttl=lit_order.ttl,
                    )
                )
            else:
                orders.append(lit_order)
        return orders

## Saver

The saver also records the executed volume of each market at each step.

In [ ]:
class DarkPoolMarketStepSaver(MarketStepSaver):
    def process_market_step_end_log(self, log: MarketStepEndLog) -> None:
        super().process_market_step_end_log(log=log)
        self.market_step_logs[-1]["executed_volume"] = log.market.get_executed_volume()

## Config

This is the config of the DarkPool sample of plhamJ.
`maxNormalOrders` is set to 2, the number of markets, which is the default of plhamJ.
The default of PAMS is 1.

In [ ]:
config = {
	"simulation": {
		"markets": ["LitMarket", "DarkPoolMarket"],
		"agents": ["DarkPoolFCNAgents"],
		"sessions": [
			{	"sessionName": 0,
				"iterationSteps": 100,
				"withOrderPlacement": True,
				"withOrderExecution": False,
				"withPrint": True,
				"maxNormalOrders": 2, "MEMO": "The same number as #markets"
			},
			{	"sessionName": 1,
				"iterationSteps": 500,
				"withOrderPlacement": True,
				"withOrderExecution": True,
				"withPrint": True,
				"maxNormalOrders": 2, "MEMO": "The same number as #markets"
			}
		]
	},

	"LitMarket": {
		"class": "Market",
		"tickSize": 0.00001,
		"marketPrice": 300.0,
		"outstandingShares": 25000
	},

	"DarkPoolMarket": {
		"class": "DarkPoolMarket",
		"tickSize": 0.00001,
		"marketPrice": 300.0,
		"outstandingShares": 25000,
		"markets": ["LitMarket"]
	},

	"DarkPoolFCNAgents": {
		"class": "DarkPoolFCNAgent",
		"numAgents": 100,

		"MEMO": "Agent class",
		"markets": ["LitMarket", "DarkPoolMarket"],
		"assetVolume": 50,
		"cashAmount": 10000,

		"MEMO": "FCNAgent class",
		"fundamentalWeight": {"expon": [1.0]},
		"chartWeight": {"expon": [0.0]},
		"noiseWeight": {"expon": [1.0]},
		"noiseScale": 0.001,
		"timeWindowSize": [100, 200],
		"orderMargin": [0.0, 0.1],

		"MEMO": "DarkPoolFCNAgent class",
		"darkPoolChance": 0.3
	}
}

## Simulation

`run` runs a simulation and returns the time series of each market.
The trade price at a step is the market price of the dark pool if the dark pool executed some orders at the step, otherwise that of the lit market if the lit market did, and otherwise NaN, as in the output of plhamJ.

In [ ]:
def run(config: Dict[str, Any], seed: int) -> Dict[str, np.ndarray]:
    saver = DarkPoolMarketStepSaver()
    runner = SequentialRunner(settings=config, prng=random.Random(seed), logger=saver)
    runner.class_register(cls=DarkPoolMarket)
    runner.class_register(cls=DarkPoolFCNAgent)
    runner.main()

    result: Dict[str, np.ndarray] = {}
    for market_name, prefix in [("LitMarket", "lit"), ("DarkPoolMarket", "dark")]:
        logs = sorted(
            filter(lambda x: x["market_name"] == market_name, saver.market_step_logs),
            key=lambda x: x["market_time"],
        )
        result["time"] = np.array([x["market_time"] for x in logs])
        result["session_id"] = np.array([x["session_id"] for x in logs])
        result[f"{prefix}_price"] = np.array([x["market_price"] for x in logs], dtype=float)
        result[f"{prefix}_fundamental"] = np.array([x["fundamental_price"] for x in logs], dtype=float)
        result[f"{prefix}_volume"] = np.array([x["executed_volume"] for x in logs], dtype=int)
    result["trade_price"] = np.where(
        result["dark_volume"] > 0,
        result["dark_price"],
        np.where(result["lit_volume"] > 0, result["lit_price"], np.nan),
    )
    return result

In [ ]:
result = run(config=config, seed=42)

## Prices

This plot corresponds to `plot.R` of plhamJ.
The trade price keeps its last value at the steps without executions, and starts from the initial market price, 300.
In the session without order execution, the market prices do not change.

In [ ]:
trade_price_filled = np.empty_like(result["trade_price"])
last_trade_price = 300.0
for i, trade_price in enumerate(result["trade_price"]):
    if not np.isnan(trade_price):
        last_trade_price = trade_price
    trade_price_filled[i] = last_trade_price

plt.plot(result["time"], result["lit_price"], color="gold", label="lit market price")
plt.plot(result["time"], result["dark_price"], color="blue", label="dark pool price")
plt.plot(result["time"], result["lit_fundamental"], color="black", label="fundamental price")
plt.plot(result["time"], trade_price_filled, color="red", label="trade price")
plt.ylim(280, 320)
plt.xlabel("t")
plt.ylabel("Price")
plt.legend()
plt.show()

## Trade volumes

This plot corresponds to `plot-tradevolume.R` of plhamJ: the centered moving sum of the executed volume over 100 steps.

In [ ]:
window = 100
moving_sum_time = result["time"][window // 2 - 1 : len(result["time"]) - window // 2]
plt.plot(
    moving_sum_time,
    np.convolve(result["lit_volume"], np.ones(window), mode="valid"),
    color="gold",
    label="lit market",
)
plt.plot(
    moving_sum_time,
    np.convolve(result["dark_volume"], np.ones(window), mode="valid"),
    color="blue",
    label="dark pool",
)
plt.xlim(0, len(result["time"]))
plt.ylim(0, 100)
plt.xlabel("t")
plt.ylabel("Volume")
plt.legend()
plt.show()

## Summary

`summarize` computes the following in the session with order execution:
- the share of the dark pool in the executed volume,
- the mean absolute deviation of the lit market price from the fundamental price, in percent, as a measure of market efficiency,
- the standard deviation of the log returns of the lit market price, in percent, as the volatility.

In [ ]:
def summarize(result: Dict[str, np.ndarray]) -> Dict[str, float]:
    in_session = result["session_id"] == 1
    lit_volume = int(result["lit_volume"][in_session].sum())
    dark_volume = int(result["dark_volume"][in_session].sum())
    lit_prices = result["lit_price"][in_session]
    fundamental_prices = result["lit_fundamental"][in_session]
    return {
        "lit_volume": lit_volume,
        "dark_volume": dark_volume,
        "dark_share": dark_volume / max(lit_volume + dark_volume, 1),
        "deviation": 100 * float(np.mean(np.abs(lit_prices - fundamental_prices) / fundamental_prices)),
        "volatility": 100 * float(np.std(np.diff(np.log(lit_prices)))),
    }


for key, value in summarize(result).items():
    print(f"{key}: {value:.4g}")

## Usage of the dark pool

Mizuta et al. (2015) study how the usage of the dark pool affects the market.
The cell below runs the simulation with some values of `darkPoolChance` and a few seeds.
Use more seeds and longer sessions for reliable results.

In [ ]:
dark_pool_chances = [0.0, 0.2, 0.4, 0.6, 0.8]
seeds = [1, 2, 3]
summaries: Dict[float, List[Dict[str, float]]] = {}
for dark_pool_chance in dark_pool_chances:
    config_sweep = copy.deepcopy(config)
    config_sweep["DarkPoolFCNAgents"]["darkPoolChance"] = dark_pool_chance
    summaries[dark_pool_chance] = [summarize(run(config=config_sweep, seed=seed)) for seed in seeds]

print("darkPoolChance  dark share  deviation [%]  volatility [%]")
for dark_pool_chance in dark_pool_chances:
    means = {key: np.mean([x[key] for x in summaries[dark_pool_chance]]) for key in ["dark_share", "deviation", "volatility"]}
    print(f"{dark_pool_chance:14.1f}  {means['dark_share']:10.3f}  {means['deviation']:13.3f}  {means['volatility']:14.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (key, label) in zip(
    axes,
    [("dark_share", "dark share"), ("deviation", "deviation [%]"), ("volatility", "volatility [%]")],
):
    values = np.array([[x[key] for x in summaries[c]] for c in dark_pool_chances])
    ax.errorbar(dark_pool_chances, values.mean(axis=1), yerr=values.std(axis=1), marker="o", capsize=3)
    ax.set_xlabel("darkPoolChance")
    ax.set_ylabel(label)
plt.tight_layout()
plt.show()

## Differences from plhamJ

- `maxNormalOrders` is set to 2 in the config, because the default of plhamJ is the number of markets while that of PAMS is 1.
- The orders to the dark pool are market orders, while plhamJ uses limit orders with the largest price. The dark pool rejects limit orders.
- plhamJ ignores the orders to the dark pool while the dark pool is not running. The dark pool here places such orders and cancels them at once.
- plhamJ sends an order to the dark pool only if the mid price of the dark pool is not finite, which is always true. The condition is omitted.
- PAMS has no itayose at the start of a session, so the orders placed in the first session are executed at the first order of the second session.
- The market price of PAMS is the last executed price even at the steps without executions, while plhamJ uses the mid price at such steps.
- `FCNAgent` of PAMS uses `timeWindowSize` for the time to live of its orders and for the expected price, while that of plhamJ uses `min(t, timeWindowSize)`. So the agents of plhamJ do not order at t = 0, and their first orders expire sooner.